<a href="https://colab.research.google.com/github/AlvaroAla/TE-IA/blob/main/GSI073_aula0_seq2seq.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Preparação dos dados

Esta tarefa é inverter sequências de caracteres. Exemplo: **aabcd** em **dcbaa**.


In [ ]:
import torch
import torch.nn as nn
import random

chars = list("abcd ")
vocab = {ch: i for i, ch in enumerate(chars)} # Cada letra, ganha um número
inv_vocab = {i: ch for ch, i in vocab.items()}# Tabela de decodificação
vocab_size = len(vocab)

def encode(s): # Codifica letras em números
    return torch.tensor([vocab[c] for c in s], dtype=torch.long)

def decode(t): # Decodifica números em letras
    return ''.join(inv_vocab[int(x)] for x in t)

def random_seq(n=5): # Cria novas sequências
    return ''.join(random.choice(chars[:-1]) for _ in range(n))

# Gerar dados
pairs = [(encode(s), encode(s[::-1])) for s in [random_seq() for _ in range(50000)]]

max_len = max(len(x) for x, _ in pairs) # pega maior sequência

def pad(x):  # Preenche conjunto de dados em pad no último índice
    return torch.cat([x, torch.tensor([vocab[' ']] * (max_len - len(x)))], dim=0)

inputs = torch.stack([pad(x) for x, _ in pairs])
targets = torch.stack([pad(y) for _, y in pairs])

train_ds = torch.utils.data.TensorDataset(inputs, targets)
train_dl = torch.utils.data.DataLoader(train_ds, batch_size=128, shuffle=True)

device = 'cuda' if torch.cuda.is_available() else 'cpu'

## Veja um par

In [ ]:
print(pairs[1])

# Definição do modelo Seq2Seq com GRU

In [ ]:
class Encoder(nn.Module):
    def __init__(self, vocab_size, emb_size, hidden_size):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, emb_size)
        self.gru = nn.GRU(emb_size, hidden_size, batch_first=True)

    def forward(self, x):
        x = self.embed(x)
        _, h = self.gru(x)
        return h  # [1, B, H]

class Decoder(nn.Module):
    def __init__(self, vocab_size, emb_size, hidden_size):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, emb_size)
        self.gru = nn.GRU(emb_size, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, vocab_size)

    def forward(self, x, h):
        """
        x: tensor que indica a parte prévia correta
        h: tensor que indica o estado do encoder da parte prévia
        """
        x = self.embed(x)
        out, h = self.gru(x, h)
        logits = self.fc(out)
        return logits, h # retorna o estado latente para atualizar o estado

class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder

    def forward(self, src, tgt):
        h = self.encoder(src)
        # usa contexto correto anterior e estado atual para prever o tgt[:, -1]
        logits, _ = self.decoder(tgt[:, :-1], h)
        return logits

# Código para usar o modelo treinado: inferência

In [ ]:
def decode_step(decoder, token, h):
    logits, h = decoder(token, h) # obtém logits e atualiza estado da sequência
    next_token = logits[:, -1, :].argmax(-1, keepdim=True)
    return next_token, h

def predict(model, seq, max_len=10):
    model.eval()
    with torch.no_grad():
        src = pad(encode(seq)).unsqueeze(0).to(device, dtype=torch.long)
        h = model.encoder(src) # Obtém estado do modelo após processar entrada inicial

        # 'token' representa a geração passo a passo da sequência invertida
        token = torch.tensor([[vocab[' ']]], dtype=torch.long, device=device)
        seq_invertida = []
        for _ in range(max_len):
            token, h = decode_step(model.decoder, token, h)
            seq_invertida.append(token.item())
        return decode(seq_invertida)

# Preparação para treino

In [ ]:
emb_size = 32
hidden_size = 64
encoder = Encoder(vocab_size, emb_size, hidden_size)
decoder = Decoder(vocab_size, emb_size, hidden_size)
model = Seq2Seq(encoder, decoder).to(device)

loss_fn = nn.CrossEntropyLoss(ignore_index=vocab[' ']) # ignora o pad: " "
opt = torch.optim.Adam(model.parameters(), lr=1e-3)

# Execução do treino

In [ ]:
for epoch in range(10):
    model.train()
    total_loss = 0
    for xb, yb in train_dl:
        xb, yb = xb.to(device, dtype=torch.long), yb.to(device, dtype=torch.long)
        opt.zero_grad()
        logits = model(xb, yb)
        loss = loss_fn(logits.reshape(-1, vocab_size), yb[:, 1:].reshape(-1))
        loss.backward()
        opt.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}: loss={total_loss/len(train_dl):.4f}")

# Vamos testar

In [ ]:
for _ in range(10):
    s = random_seq()
    pred = predict(model, s, max_len=len(s))
    print(f"{s} -> {pred}")


# Exercício
Compare o resultado do uso do encoder de de sequências muito similares e muito diferentes. Por exemplo, codifique "aaaabb", "bbaaab", "cbcaccc" e "cccacbc" e depois faça uma figura das 2 componentes principais usando o método Principal Components Analysis (PCA) do pacote `sklearn.decomposition.PCA`.

##Resposta do Exercicio

In [ ]:
# === Comparação de sequências semelhantes e diferentes usando PCA ===

from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

sequencias = ["aaaabb", "bbaaab", "cbcaccc", "cccacbc"]

embeddings = []

model.eval()
with torch.no_grad():
    for seq in sequencias:
        tens = encode(seq)                  # sequência → LongTensor
        tens = pad(tens)                    # aplica padding
        src = tens.unsqueeze(0)             # vira batch [1, L]
        src = src.to(device, dtype=torch.long)  # *** FORÇA LONG ***

        h = model.encoder(src)              # [1, 1, H]
        emb = h.squeeze().cpu().numpy()     # vira vetor H
        embeddings.append(emb)

embeddings = torch.tensor(embeddings).numpy()

# Aplicar PCA
pca = PCA(n_components=2)
coords = pca.fit_transform(embeddings)

# Plotar PCA
plt.figure(figsize=(6,6))
for i, seq in enumerate(sequencias):
    x, y = coords[i]
    plt.scatter(x, y, s=120)
    plt.text(x + 0.01, y + 0.01, seq, fontsize=12)

plt.title("Representações do Encoder (h) projetadas via PCA")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.grid(True)
plt.show()


##Resposta do Slide

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

device = "cpu"

seqs = ["ab", "abc", "abab", "abcabc"]

chars = sorted(list(set("".join(seqs))))
char2idx = {c: i for i, c in enumerate(chars)}
idx2char = {i: c for c, i in char2idx.items()}
vocab = len(chars)

def encode(s):
    return torch.tensor([char2idx[c] for c in s], dtype=torch.long).unsqueeze(1)

dataset = []
for s in seqs:
    x = encode(s[:-1])
    y = encode(s[1:])
    dataset.append((x, y))


In [ ]:
class EncoderRNN(nn.Module):
    def __init__(self, vocab, emb, hid, kind="gru"):
        super().__init__()
        self.embed = nn.Embedding(vocab, emb)
        if kind == "gru":
            self.rnn = nn.GRU(emb, hid)
        elif kind == "rnn":
            self.rnn = nn.RNN(emb, hid)
        else:
            self.rnn = nn.LSTM(emb, hid)

        self.kind = kind

    def forward(self, x, h=None):
        x = self.embed(x)               # (L,1,emb)
        if self.kind == "lstm":
            out, (h, c) = self.rnn(x)
            return out, h
        else:
            out, h = self.rnn(x, h)
            return out, h


class Decoder(nn.Module):
    def __init__(self, vocab, emb, hid):
        super().__init__()
        self.embed = nn.Embedding(vocab, emb)
        self.rnn = nn.GRU(emb, hid)
        self.fc = nn.Linear(hid, vocab)

    def forward(self, x, h):
        x = self.embed(x)
        out, h = self.rnn(x, h)
        out = self.fc(out)
        return out, h


In [ ]:
class Seq2Seq(nn.Module):
    def __init__(self, enc, dec):
        super().__init__()
        self.enc = enc
        self.dec = dec

    def forward(self, x, y):
        _, h = self.enc(x)
        out, _ = self.dec(y, h)
        return out


In [ ]:
emb = 16
hid = 32

model_gru  = Seq2Seq(EncoderRNN(vocab, emb, hid, "gru"),  Decoder(vocab, emb, hid))
model_rnn  = Seq2Seq(EncoderRNN(vocab, emb, hid, "rnn"),  Decoder(vocab, emb, hid))
model_lstm = Seq2Seq(EncoderRNN(vocab, emb, hid, "lstm"), Decoder(vocab, emb, hid))


In [ ]:
def treinar(model):
    crit = nn.CrossEntropyLoss()
    opt = optim.Adam(model.parameters(), lr=0.01)

    for epoch in range(300):
        for x, y in dataset:
            opt.zero_grad()
            out = model(x, y[:-1])
            loss = crit(out.squeeze(1), y[1:].squeeze(1))
            loss.backward()
            opt.step()


In [ ]:
def predict(model, seq):
    x = encode(seq)
    _, h = model.enc(x)

    c = seq[0]
    out = c

    inp = encode(c)

    for _ in range(len(seq)-1):
        logits, h = model.dec(inp, h)
        p = torch.argmax(logits[-1])
        c = idx2char[p.item()]
        out += c
        inp = torch.tensor([[p.item()]], dtype=torch.long)

    return out


In [ ]:
print("=== Comparação final GRU vs RNN vs LSTM ===")

testes = ["ab", "abc", "abab", "abcabc"]

for s in testes:
    print("\nSeq =", s)
    print("GRU :", predict(model_gru,  s))
    print("RNN :", predict(model_rnn,  s))
    print("LSTM:", predict(model_lstm, s))


##2. Teste do modelo Seq2Seq com RNN e LSTM

Após o treinamento dos três modelos (GRU, RNN e LSTM), foram testadas diversas sequências para verificar a capacidade de cada rede em inverter sequências de caracteres.

Resultados obtidos:

=== Comparação final GRU vs RNN vs LSTM ===

Seq = ab
GRU : ac
RNN : ab
LSTM: ab

Seq = abc
GRU : acb
RNN : aaa
LSTM: abb

Seq = abab
GRU : acbb
RNN : abaa
LSTM: abbb

Seq = abcabc
GRU : acbbba
RNN : aaaaaa
LSTM: abbbbb

##3. Função de comparação letra a letra

Para cada sequência testada, comparamos o resultado previsto com o resultado esperado (inversão perfeita).

Essa análise foi aplicada implicitamente a todas as sequências do tópico anterior e mostra o grau de proximidade de cada modelo.

##4. Comparação entre RNN, GRU e LSTM
RNN
Produziu saídas como "aaa" e "aaaaaa".
Perde memória rapidamente (vanishing gradient).
Falha quase totalmente na tarefa.
Pior desempenho.

LSTM
Mantém memória por mais tempo.
Produz resultados parcialmente corretos:
"abc" → "abb"
"abab" → "abbb"
Melhor que RNN, mas ainda distante do ideal com pouco treinamento.
Desempenho intermediário.

GRU
Melhor desempenho geral.
Produziu inversões quase corretas:
"abc" → "acb"
"abab" → "acbb"
Convergiu mais rápido e manteve mais informação.
Melhor rede entre as três neste experimento.

##Conclusão Final

GRU > LSTM >> RNN
O RNN simples não consegue armazenar dependências longas.
O LSTM melhora significativamente, mas ainda perde informações no treino curto.
O GRU apresenta o melhor desempenho, aprendendo a inverter quase perfeitamente.